In [1]:
import os
import time
import pandas as pd
import pickle
from tqdm import tqdm
from nba_api.stats.static import players
from nba_api.stats.endpoints import playercareerstats
import random

In [2]:
# Function to handle API calls with retries
def safe_api_call(api_function, *args, **kwargs):
    """Retries API calls with exponential backoff if rate limited."""
    retries = 5
    delay = 1  # Start with 1-second delay

    for attempt in range(retries):
        try:
            return api_function(*args, **kwargs)  # Call API function
        except Exception as e:
            print(f"API Error: {e}. Retrying in {delay:.1f}s...")
            time.sleep(delay)
            delay *= 2 + random.uniform(0, 1)  # Exponential backoff with randomness
    print("Max retries reached. Skipping request.")
    return None

In [5]:
# File paths
csv_file = "nba_career_stats.csv"
cache_file = "nba_api_cache.pkl"

# Load existing data if available
if os.path.exists(csv_file):
    print("Loading existing data from CSV...")
    career_stats_df = pd.read_csv(csv_file) 
    processed_players = set(career_stats_df["Player"].unique())  # Track players already fetched
else:
    print("No existing CSV found. Starting fresh...")
    career_stats_df = pd.DataFrame()
    processed_players = set()
# Load API cache if available
if os.path.exists(cache_file):
    with open(cache_file, "rb") as f:
        api_cache = pickle.load(f)
else:
    api_cache = {}

Loading existing data from CSV...


In [4]:
# Get all NBA players
nba_players = players.get_players()

# Process players in small batches
batch_size = 50

for i in range(0, len(nba_players), batch_size):
    batch = nba_players[i : i + batch_size]

    for player in tqdm(batch, desc="Fetching Player Stats"):
        player_name = player["full_name"]
        player_id = player["id"]

        # Skip players already processed
        if player_name in processed_players:
            continue

        # Check cache first
        if player_id in api_cache:
            career = api_cache[player_id]
        else:
            career = safe_api_call(playercareerstats.PlayerCareerStats, player_id=player_id)
            api_cache[player_id] = career
            with open(cache_file, "wb") as f:
                pickle.dump(api_cache, f)  # Save updated cache

        if career:
            df = career.get_data_frames()[0]
            df["Player"] = player_name

            # Assign season numbers
            df["YearsExperience"] = range(len(df))  # Season 0, 1, 2, etc.

            # Use pd.concat() to append new data
            career_stats_df = pd.concat([career_stats_df, df], ignore_index=True)

        # Avoid rate limiting
        time.sleep(1.5)  # Increased delay to prevent blocking

    # Save progress after each batch
    career_stats_df.to_csv(csv_file, index=False)
    print("✅ Saved progress to CSV.")

# Final save
career_stats_df.to_csv(csv_file, index=False)
print("✅ Final save to CSV.")

# Display the first few rows
print(career_stats_df.head())

Fetching Player Stats: 100%|██████████| 50/50 [00:00<?, ?it/s]


✅ Saved progress to CSV.


Fetching Player Stats: 100%|██████████| 50/50 [00:00<?, ?it/s]


✅ Saved progress to CSV.


Fetching Player Stats: 100%|██████████| 50/50 [00:00<?, ?it/s]


...

API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 1.0s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 2.8s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 6.4s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 16.9s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 44.3s...
Max retries reached. Skipping request.


Fetching Player Stats:  86%|████████▌ | 43/50 [47:35<24:47, 212.47s/it]

API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 1.0s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 2.1s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 5.2s...
API Error: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30). Retrying in 14.4s...


Fetching Player Stats:  86%|████████▌ | 43/50 [50:28<08:12, 70.43s/it] 


KeyboardInterrupt: 

In [6]:
career_stats_df

,PLAYER_ID,SEASON_ID,LEAGUE_ID,TEAM_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,Player,YearsExperience
0,76001,1990-91,0,1610612757,POR,23.0,43,0.0,290.0,55,...,62.0,89.0,12,4.0,12.0,22.0,39,135,Alaa Abdelnaby,0
1,76001,1991-92,0,1610612757,POR,24.0,71,1.0,934.0,178,...,179.0,260.0,30,25.0,16.0,66.0,132,432,Alaa Abdelnaby,1
2,76001,1992-93,0,1610612749,MIL,25.0,12,0.0,159.0,26,...,25.0,37.0,10,6.0,4.0,13.0,24,64,Alaa Abdelnaby,2
3,76001,1992-93,0,1610612738,BOS,25.0,63,52.0,1152.0,219,...,186.0,300.0,17,19.0,22.0,84.0,165,514,Alaa Abdelnaby,3
4,76001,1992-93,0,0,TOT,25.0,75,52.0,1311.0,245,...,211.0,337.0,27,25.0,26.0,97.0,189,578,Alaa Abdelnaby,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18560,371,1996-97,0,1610612765,DET,29.0,79,5.0,1999.0,312,...,309.0,377.0,99,35.0,27.0,85.0,161,857,Terry Mills,8
18561,371,1997-98,0,1610612748,MIA,30.0,50,0.0,782.0,81,...,118.0,152.0,39,19.0,9.0,45.0,129,212,Terry Mills,9
18562,371,1998-99,0,1610612748,MIA,31.0,1,0.0,29.0,3,...,1.0,4.0,0,1.0,0.0,3.0,3,9,Terry Mills,10
18563,371,1999-00,0,1610612765,DET,32.0,82,78.0,1842.0,214,...,340.0,390.0,85,38.0,24.0,46.0,242,548,Terry Mills,11
